# Fase 1A — Preprocesamiento para Machine Learning (IroSvA)

Este notebook prepara las entradas utilizadas por LR, RF y XGBoost.

Se generan tres representaciones del texto:

- `normal`;
- `stem`;
- `lemma`.

Se mantiene el control de calidad interno del train (duplicados exactos y conflictos de etiqueta) y la extracción de las 9 variables lingüísticas.

El test se procesa con las mismas reglas, pero **no se utiliza para eliminar registros del train**.

La prevención de leakage entre folds se realiza posteriormente, en los notebooks de modelos, mediante grupos de textos coincidentes + `StratifiedGroupKFold`.

## 1. Librerías y entorno

Importaciones y versiones de los paquetes utilizados.

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import emoji
import spacy
from nltk.stem import SnowballStemmer
from tqdm import tqdm
from IPython.display import display
import nltk

VERSIONES = {
    "pandas": pd.__version__,
    "emoji": emoji.__version__,
    "spacy": spacy.__version__,
    "nltk": nltk.__version__,
}
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))

## 2. Configuración

Las rutas, las variantes dialectales y el modelo de spaCy se definen una sola vez en esta celda. Los CSV se exportan en `../data/`.

In [ ]:
DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANTES = ["mx", "es", "cu"]
SPACY_MODELO = "es_core_news_sm"

print(f"[CONFIG][Preprocesamiento ML] variantes={VARIANTES} | modelo spaCy={SPACY_MODELO}")
print(f"[CONFIG][Preprocesamiento ML] entrada={DATA_DIR.resolve()}")
print(f"[CONFIG][Preprocesamiento ML] salida={OUT_DIR.resolve()}")

## 3. Carga y auditoría del train oficial

Se cargan las tres variantes dialectales por separado.

Antes de eliminar registros se verifica, comparando el `MESSAGE` original de forma exacta:

- cuántos textos `MESSAGE` están repetidos;
- si un mismo texto repetido tiene **la misma etiqueta**;
- si existe un **conflicto de anotación** (mismo `MESSAGE` con etiquetas distintas).

Después de la auditoría:

1. los textos con etiquetas contradictorias se retiran **completamente** del train y se guardan en `conflictos_etiqueta_train_ml.csv`;
2. de los duplicados restantes (misma etiqueta) se conserva una sola copia (la primera).

El test no interviene en este control. El mismo criterio se aplica en los preprocesamientos de ML y de Transformers.

In [ ]:
def auditar_duplicados_train(df, variante):
    """Cuenta textos repetidos y conflictos de etiqueta dentro del train de una variante."""
    grupos = df.groupby("MESSAGE", dropna=False)["IS_IRONIC"].agg(["size", "nunique"]).reset_index()
    repetidos = grupos[grupos["size"] > 1]
    conflictos = repetidos[repetidos["nunique"] > 1]
    mismas_etiquetas = repetidos[repetidos["nunique"] == 1]

    return {
        "variante": variante,
        "n_original": len(df),
        "n_textos_repetidos": len(repetidos),
        "n_duplicados_misma_etiqueta": len(mismas_etiquetas),
        "n_conflictos_etiqueta": len(conflictos),
        "mensajes_conflicto": set(conflictos["MESSAGE"].tolist()),
    }


def cargar_train():
    """Carga el train de cada variante, retira conflictos y deduplica textos exactos."""
    frames = []
    auditoria = []
    conflictos_detalle = []

    for variante in VARIANTES:
        df = pd.read_csv(DATA_DIR / f"irosva.{variante}.training.csv")
        df["VARIANTE"] = variante

        info = auditar_duplicados_train(df, variante)
        mensajes_conflicto = info["mensajes_conflicto"]

        # Guardar TODAS las filas implicadas en conflictos.
        df_conflictos = df[df["MESSAGE"].isin(mensajes_conflicto)].copy()
        if not df_conflictos.empty:
            df_conflictos["MOTIVO_EXCLUSION"] = "mismo_texto_etiquetas_contradictorias"
            conflictos_detalle.append(df_conflictos)
            print(f"\n{variante.upper()} — conflictos de etiqueta:")
            display(
                df_conflictos[["ID", "MESSAGE", "IS_IRONIC", "VARIANTE"]].sort_values("MESSAGE")
            )

        # 1. Retirar completamente los textos con etiquetas contradictorias.
        df_sin_conflictos = df[~df["MESSAGE"].isin(mensajes_conflicto)].copy()

        # 2. En los duplicados restantes la etiqueta coincide: conservar una copia.
        df_limpio = df_sin_conflictos.drop_duplicates(subset="MESSAGE", keep="first").reset_index(
            drop=True
        )
        n_duplicados_retirados = len(df_sin_conflictos) - len(df_limpio)

        auditoria.append(
            {
                "variante": variante,
                "n_original": info["n_original"],
                "textos_repetidos": info["n_textos_repetidos"],
                "conflictos_texto": info["n_conflictos_etiqueta"],
                "filas_retiradas_conflicto": len(df_conflictos),
                "filas_duplicadas_misma_etiqueta_retiradas": n_duplicados_retirados,
                "n_tras_control_duplicados": len(df_limpio),
            }
        )

        print(
            f"[TRAIN][{variante.upper()}] {info['n_original']} originales | "
            f"{len(df_conflictos)} retirados por conflicto | "
            f"{n_duplicados_retirados} duplicados de misma etiqueta retirados | "
            f"{len(df_limpio)} restantes"
        )
        frames.append(df_limpio)

    df_train = pd.concat(frames, ignore_index=True)
    df_auditoria = pd.DataFrame(auditoria)
    df_conflictos_train = (
        pd.concat(conflictos_detalle, ignore_index=True) if conflictos_detalle else pd.DataFrame()
    )
    return df_train, df_auditoria, df_conflictos_train


df_train, df_auditoria_train, df_conflictos_train = cargar_train()

print("\nAUDITORÍA TRAIN — DUPLICADOS Y CONFLICTOS")
display(df_auditoria_train)

print(f"\n[TRAIN] Total tras control de duplicados/conflictos: {len(df_train)} registros")
print(df_train["VARIANTE"].value_counts().sort_index())

print("\nDistribución de clases:")
print(df_train.groupby("VARIANTE")["IS_IRONIC"].value_counts())

## 4. Carga del test oficial

El test se carga para generar las mismas representaciones que el train (`normal`, `stem` y `lemma`).

No se utiliza para modificar la composición del train.

In [ ]:
def cargar_test(variante):
    """Une los textos del test con sus etiquetas oficiales."""
    df_texto = pd.read_csv(DATA_DIR / f"irosva.{variante} - irosva.{variante}.test.csv")
    df_truth = pd.read_csv(DATA_DIR / f"irosva.{variante}.test.truth.csv")

    if "IS_IRONIC" in df_texto.columns:
        df_texto = df_texto.drop(columns=["IS_IRONIC"])

    df = df_texto.merge(df_truth[["ID", "IS_IRONIC"]], on="ID", how="inner")
    assert len(df) == len(df_texto), f"{variante}: hay textos del test sin etiqueta oficial"

    df["VARIANTE"] = variante
    return df


df_test = pd.concat([cargar_test(v) for v in VARIANTES], ignore_index=True)

print(f"[TEST] Total: {len(df_test)} registros")
print(df_test["VARIANTE"].value_counts().sort_index())

## 5. Variables lingüísticas

Se extraen 9 variables lingüísticas del texto original, antes de cualquier limpieza. Son idénticas para las tres representaciones (`normal`, `stem`, `lemma`) porque se calculan una sola vez sobre el texto sin procesar.

(González-Ibáñez et al., 2011; Joshi et al., 2015, 2016; Šandor & Bagić Babac, 2023; RAE, 2011)

In [ ]:
def quitar_tildes(texto):
    """Convierte caracteres acentuados a su forma base para capturar
    variantes de escritura informal (jamás→jamas, ningún→ningun)."""
    return "".join(
        c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn"
    )


# Negaciones en español.
# Fuente: RAE y Asociación de Academias de la Lengua Española (2011).
# Nueva gramática básica de la lengua española.
NEGACIONES = {
    "no",
    "nunca",
    "jamas",
    "tampoco",
    "nada",
    "nadie",
    "ningun",
    "ninguno",
    "ninguna",
    "ningunos",
    "ningunas",
    "ni",
    "sino",
    "sin",
}

FEATURE_COLS = ["n_exc", "n_int", "n_may", "n_emo", "n_ris", "n_neg", "n_elo", "n_com", "n_pun"]


def extraer_features_linguisticos(texto):
    """Extrae 9 variables lingüísticas como conteos sobre el texto original."""
    texto_str = str(texto)
    texto_norm = quitar_tildes(texto_str.lower())
    palabras = re.findall(r"\b\w+\b", texto_norm)

    # 1. Exclamaciones (González-Ibáñez et al., 2011; Joshi et al., 2015)
    n_exc = texto_str.count("!") + texto_str.count("¡")
    # 2. Interrogaciones (González-Ibáñez et al., 2011; Joshi et al., 2015)
    n_int = texto_str.count("?") + texto_str.count("¿")
    # 3. Mayúsculas enfáticas: 3+ caracteres (Joshi et al., 2015; Šandor & Bagić Babac, 2023)
    n_may = len(re.findall(r"\b[A-ZÁÉÍÓÚÜÑ]{3,}\b", texto_str))
    # 4. Emojis (González-Ibáñez et al., 2011; Joshi et al., 2015)
    n_emo = sum(1 for c in texto_str if c in emoji.EMOJI_DATA)
    # 5. Risas (González-Ibáñez et al., 2011)
    n_ris = len(re.findall(r"\b(ja+ja+|je+je+|ji+ji+|ha+ha+|xd+|lol+|jsjs)\b", texto_norm))
    # 6. Negaciones (González-Ibáñez et al., 2011; RAE, 2011)
    n_neg = sum(1 for p in palabras if p in NEGACIONES)
    # 7. Elongación: mismo carácter 3+ veces (Šandor & Bagić Babac, 2023; Joshi et al., 2016)
    n_elo = len(re.findall(r"(.)\1{2,}", texto_norm))
    # 8. Comillas (González-Ibáñez et al., 2011; Šandor & Bagić Babac, 2023)
    n_com = len(re.findall(r'["\'\u2018\u2019\u201c\u201d«»]', texto_str))
    # 9. Puntos suspensivos (Šandor & Bagić Babac, 2023; Joshi et al., 2016)
    n_pun = texto_str.count("...") + texto_str.count("\u2026")

    return [n_exc, n_int, n_may, n_emo, n_ris, n_neg, n_elo, n_com, n_pun]

In [ ]:
for nombre, df in [("Train", df_train), ("Test", df_test)]:
    features = df["MESSAGE"].fillna("").apply(extraer_features_linguisticos).tolist()
    for i, col in enumerate(FEATURE_COLS):
        df[col] = [f[i] for f in features]

    print(f"\n[FEATURES][{nombre.upper()}] Medias:")
    print(df[FEATURE_COLS].mean().round(3))

## 6. Preprocesamiento base

Genera `MESSAGE_CLEAN` a partir del texto original:

- URL → `[URL]` y mención → `[USER]` (tras pasar a minúsculas quedan como `[url]` y `[user]`);
- emojis convertidos a su descripción en español;
- caracteres repetidos 3 o más veces reducidos a 2;
- eliminación de la puntuación (incluidos `¿?`, `¡!` y `#`);
- minúsculas y espacios normalizados.

Las tildes se conservan. Este texto es la base de las tres representaciones.

In [ ]:
def preprocesar_ml_base(texto):
    """Limpieza de texto: URLs, menciones, emojis a texto,
    normalización de elongación, eliminación de puntuación, minúsculas."""
    texto = re.sub(r"http\S+|www\S+", "[URL]", texto)
    texto = re.sub(r"@\w+", "[USER]", texto)
    texto = emoji.demojize(texto, language="es")
    texto = re.sub(r"(.)\1{2,}", r"\1\1", texto)
    texto = re.sub(r"[^\w\s\[\]áéíóúüñÁÉÍÓÚÜÑ]", " ", texto)
    texto = texto.lower()
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

## 7. Stemming

Aplica `SnowballStemmer("spanish")` sobre `MESSAGE_CLEAN` para obtener `MESSAGE_STEM`.

In [ ]:
stemmer = SnowballStemmer("spanish")


def aplicar_stemming(texto):
    """Reduce cada palabra a su raíz con Snowball."""
    return " ".join(stemmer.stem(p) for p in str(texto).split())

## 8. Lematización

Aplica lematización con spaCy (`es_core_news_sm`) sobre `MESSAGE_CLEAN` para obtener `MESSAGE_LEMMA`.

Si no tienes el modelo instalado ejecuta:
`!python -m spacy download es_core_news_sm`

In [ ]:
nlp = spacy.load(SPACY_MODELO, disable=["parser", "ner"])
print(f"Modelo spaCy cargado: {SPACY_MODELO}")


def lematizar_corpus(textos, batch_size=256):
    """Lematiza una lista de textos con spaCy, en lotes."""
    lemas = []
    for doc in tqdm(nlp.pipe(textos, batch_size=batch_size), total=len(textos), desc="Lematizando"):
        lemas.append(" ".join(t.lemma_ for t in doc if not t.is_space))
    return lemas

## 9. Generación de textos para ML

Se aplican las tres representaciones al train y al test, con las mismas reglas.

In [ ]:
for nombre, df in [("Train", df_train), ("Test", df_test)]:
    print(f"Procesando {nombre}...")
    df["MESSAGE_CLEAN"] = df["MESSAGE"].fillna("").astype(str).apply(preprocesar_ml_base)
    df["MESSAGE_STEM"] = df["MESSAGE_CLEAN"].apply(aplicar_stemming)
    df["MESSAGE_LEMMA"] = lematizar_corpus(df["MESSAGE_CLEAN"].tolist())

print("\nEjemplos por variante:")
for variante in VARIANTES:
    row = df_train[df_train["VARIANTE"] == variante].iloc[0]
    print(f"\n--- {variante.upper()} ---")
    print("ORIGINAL:", row["MESSAGE"])
    print("NORMAL:  ", row["MESSAGE_CLEAN"])
    print("STEM:    ", row["MESSAGE_STEM"])
    print("LEMMA:   ", row["MESSAGE_LEMMA"])

## 10. Exportación final por variante

Se exportan 18 archivos:

- 3 representaciones (`normal`, `stem`, `lemma`);
- 3 variantes dialectales (`mx`, `es`, `cu`);
- 2 splits (`train`, `test`).

Los tres archivos de train de una misma variante contienen las mismas filas y solo cambia `MESSAGE_CLEAN`. También se guardan la auditoría del train, las filas retiradas por conflicto y un manifest con el número de filas por variante.

El preprocesamiento no elimina registros del train en función del contenido del test.

In [ ]:
COLUMNAS_BASE = ["ID", "TOPIC", "IS_IRONIC", "MESSAGE", "VARIANTE"] + FEATURE_COLS

# Columna generada -> sufijo del archivo exportado.
REPRESENTACIONES = {"MESSAGE_CLEAN": "", "MESSAGE_STEM": "_stem", "MESSAGE_LEMMA": "_lemma"}


def exportar_por_variante(df, split, col_texto, sufijo):
    """Exporta un CSV por variante; el texto elegido se guarda siempre como MESSAGE_CLEAN."""
    salida_total = df[COLUMNAS_BASE].copy()
    salida_total["MESSAGE_CLEAN"] = df[col_texto]

    for variante in VARIANTES:
        salida = salida_total[salida_total["VARIANTE"] == variante].reset_index(drop=True)
        ruta = OUT_DIR / f"{split}_clean{sufijo}_{variante}.csv"
        salida.to_csv(ruta, index=False)
        print(f"[EXPORT][{split.upper()}][{variante.upper()}] {len(salida)} registros -> {ruta}")


print("=" * 78)
print("EXPORTACIÓN FINAL — ML")
print("=" * 78)

for col_texto, sufijo in REPRESENTACIONES.items():
    for split, df in [("train", df_train), ("test", df_test)]:
        exportar_por_variante(df, split, col_texto, sufijo)


# Auditoría interna del train.
df_auditoria_train.to_csv(OUT_DIR / "auditoria_duplicados_conflictos_train_ml.csv", index=False)

if len(df_conflictos_train) > 0:
    df_conflictos_train.to_csv(OUT_DIR / "conflictos_etiqueta_train_ml.csv", index=False)


# Manifest mínimo: permite comprobar versión y cantidad de filas.
manifest = pd.DataFrame(
    [
        {
            "pipeline_version": "ml_comun_v1",
            "variante": variante,
            "train_final": int((df_train["VARIANTE"] == variante).sum()),
            "test_n": int((df_test["VARIANTE"] == variante).sum()),
        }
        for variante in VARIANTES
    ]
)
manifest.to_csv(OUT_DIR / "manifest_preprocesamiento_ml.csv", index=False)

print("\nMANIFEST DEL PREPROCESAMIENTO")
display(manifest)

print("\n[PREPROCESS][ML] EXPORTACIÓN: OK")

## 11. Uso posterior

LR, RF y XGBoost cargan directamente:

```text
data/train_clean_{mx,es,cu}.csv        data/test_clean_{mx,es,cu}.csv
data/train_clean_stem_{mx,es,cu}.csv   data/test_clean_stem_{mx,es,cu}.csv
data/train_clean_lemma_{mx,es,cu}.csv  data/test_clean_lemma_{mx,es,cu}.csv
```

Los notebooks de modelos construyen después los grupos de textos coincidentes a partir de estas representaciones.

El preprocesamiento no decide los folds ni elimina registros en función del test.

## 12. Resumen metodológico

### Train

- control interno de duplicados exactos y conflictos de etiqueta;
- extracción de las 9 variables lingüísticas;
- generación de las representaciones `normal`, `stem` y `lemma`;
- exportación.

### Modelos

LR, RF y XGBoost:

1. cargan el train preprocesado;
2. construyen los grupos de textos coincidentes;
3. ejecutan `StratifiedGroupKFold` 5×3;
4. seleccionan preprocesamiento e hiperparámetros con F1-Macro.

### Test

Se mantiene como conjunto de evaluación final y no determina qué registros permanecen en train.